In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp05/ex_8.py ──
"""
Shared infrastructure for Exercise 8 — Reinforcement Learning.

Contains: CartPole setup, reward plotting helpers, ExperimentTracker/ModelRegistry
setup, custom environment base class, evaluation utilities.
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
import random
from collections import deque
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

import gymnasium as gym
from gymnasium import spaces

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
device = get_device()

# Output directory for all visualisation artifacts
OUTPUT_DIR = Path("outputs") / "ex8_reinforcement_learning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# CARTPOLE ENVIRONMENT
# ════════════════════════════════════════════════════════════════════════


def make_cartpole() -> tuple[gym.Env, int, int]:
    """Create CartPole-v1 and return (env, obs_dim, n_actions)."""
    env = gym.make("CartPole-v1")
    obs_space = env.observation_space
    act_space = env.action_space
    assert (
        isinstance(obs_space, spaces.Box) and obs_space.shape is not None
    ), f"CartPole obs space expected gymnasium Box, got {type(obs_space).__name__}"
    assert isinstance(
        act_space, spaces.Discrete
    ), f"CartPole action space expected Discrete, got {type(act_space).__name__}"
    obs_dim = obs_space.shape[0]
    n_actions = int(act_space.n)
    print(f"CartPole-v1  obs_dim={obs_dim}  n_actions={n_actions}")
    return env, obs_dim, n_actions


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_rl.db"
    registry_db = "sqlite:///mlfp05_rl_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_reinforcement_learning", registry, True


def setup_engines() -> tuple:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# REPLAY BUFFER — shared by DQN and custom env training
# ════════════════════════════════════════════════════════════════════════


class ReplayBuffer:
    """Fixed-size buffer storing (state, action, reward, next_state, done)."""

    def __init__(self, capacity: int = 10_000):
        self.buffer: deque = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        batch = random.sample(list(self.buffer), batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(np.array(states), dtype=torch.float32, device=device),
            torch.tensor(actions, dtype=torch.long, device=device),
            torch.tensor(rewards, dtype=torch.float32, device=device),
            torch.tensor(np.array(next_states), dtype=torch.float32, device=device),
            torch.tensor(dones, dtype=torch.float32, device=device),
        )

    def __len__(self):
        return len(self.buffer)


# ════════════════════════════════════════════════════════════════════════
# DQN NETWORK — shared by DQN training and custom env training
# ════════════════════════════════════════════════════════════════════════


class DQN(nn.Module):
    """Deep Q-Network: maps state -> Q-value for each action."""

    def __init__(self, obs_dim: int, n_actions: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ════════════════════════════════════════════════════════════════════════
# EVALUATION UTILITY
# ════════════════════════════════════════════════════════════════════════


def evaluate_policy(env: gym.Env, policy_fn, n_episodes: int = 30) -> list[float]:
    """Evaluate a policy function over n_episodes. Returns list of total rewards."""
    eval_returns: list[float] = []
    for i in range(n_episodes):
        state, _ = env.reset(seed=1000 + i)
        total = 0.0
        done = False
        while not done:
            action = policy_fn(state)
            state, reward, terminated, truncated, _ = env.step(action)
            total += float(reward)
            done = terminated or truncated
        eval_returns.append(total)
    return eval_returns


# ════════════════════════════════════════════════════════════════════════
# REWARD PLOTTING HELPERS
# ════════════════════════════════════════════════════════════════════════


def moving_average(xs: list[float], window: int = 10) -> list[float]:
    """Smooth a time series with a rolling mean."""
    if len(xs) < window:
        return xs
    arr = np.asarray(xs, dtype=np.float32)
    kernel = np.ones(window, dtype=np.float32) / window
    return list(np.convolve(arr, kernel, mode="valid"))


def plot_reward_curve(
    viz: ModelVisualizer,
    rewards: list[float],
    title: str,
    filename: str,
    window: int = 20,
    x_label: str = "Episode",
    y_label: str = "Reward",
) -> None:
    """Plot a reward curve with moving average and save to HTML."""
    metrics = {
        f"{title} reward": rewards,
        f"{title} moving avg ({window})": moving_average(rewards, window),
    }
    fig = viz.training_history(metrics=metrics, x_label=x_label, y_label=y_label)
    out_path = OUTPUT_DIR / filename
    fig.write_html(str(out_path))
    print(f"  Saved: {out_path}")


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION HELPER
# ════════════════════════════════════════════════════════════════════════


async def _register_rl_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    metrics_dict: dict[str, float],
):
    """Register a single RL policy network in ModelRegistry."""
    from kailash_ml.types import MetricSpec

    model_bytes = pickle.dumps(model.state_dict())
    metric_specs = [MetricSpec(name=k, value=v) for k, v in metrics_dict.items()]
    version = await registry.register_model(
        name=name,
        artifact=model_bytes,
        metrics=metric_specs,
    )
    print(f"  Registered {name}: version={version.version}")
    return version


def register_rl_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    metrics_dict: dict[str, float],
):
    """Sync wrapper for RL model registration."""
    return asyncio.run(_register_rl_model(registry, name, model, metrics_dict))




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 8.3: Custom Gymnasium Environments for Business
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this exercise, you will be able to:
#   - Explain why custom environments matter: RL is only as good as the
#     simulation it trains on
#   - Build 5 Gymnasium-compliant environments for real Singapore
#     business problems
#   - Design observation spaces, action spaces, reward functions, and
#     episode termination logic for each domain
#   - Train DQN on a custom environment and register the trained policy
#     in the ModelRegistry
#   - Visualise environment state trajectories and action distributions
#   - Evaluate learned policies against rule-based baselines
#
# PREREQUISITES: M5/ex_8/01_dqn.py and M5/ex_8/02_ppo.py.
# ESTIMATED TIME: ~40 min
# DATASETS: No static dataset — the environments ARE the data source.
#
# TASKS:
#   1. Theory: why custom environments are the foundation of applied RL
#   2. Build 5 business-themed Gymnasium environments
#   3. Train DQN on ChurnPrevention, register in ModelRegistry
#   4. Visualise: environment state trajectories, action distributions,
#      learned policy vs baseline
#   5. Apply: each environment IS the application — evaluate ChurnPrevention
#      with churn rate reduction and revenue impact metrics
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

import gymnasium as gym
from gymnasium import spaces

import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from kailash_ml import ModelVisualizer



## TASK 1 — Theory: Why Custom Environments Matter


In [ ]:
# CartPole is a toy. In the real world, RL agents don't balance poles —
# they make BUSINESS DECISIONS: pricing, inventory, resource allocation,
# customer retention, portfolio management.
#
# The environment IS the problem definition. Get it wrong, and even a
# perfect RL algorithm will learn the wrong thing. Building a good
# environment requires:
#
#   1. STATE: What information does the decision-maker observe?
#      Too little -> agent can't learn. Too much -> curse of dimensionality.
#
#   2. ACTIONS: What can the decision-maker do?
#      Too few -> can't express optimal behaviour. Too many -> slow learning.
#
#   3. REWARD: What defines success?
#      This is the HARDEST part. Reward shaping is an art:
#      - Too sparse (reward only at episode end) -> agent can't learn
#      - Too dense (reward every step) -> agent may "hack" the reward
#      - Misaligned (reward proxy, not true objective) -> Goodhart's Law
#
#   4. DYNAMICS: How does the world respond to actions?
#      Must be realistic enough to transfer to the real system.
#
# Each environment below models a REAL business problem with realistic
# dynamics calibrated to Singapore market conditions.

print("=" * 70)
print("  TASK 1: Custom Environments — The Foundation of Applied RL")
print("=" * 70)

conn, tracker, exp_name, registry, has_registry = setup_engines()



## TASK 2 — Build 5 Business-Themed Gymnasium Environments


In [ ]:
print("\n" + "=" * 70)
print("  TASK 2: Build 5 Custom Environments")
print("=" * 70)



### Environment 1: Customer Churn Prevention (Singtel / StarHub)


SCENARIO: You manage the retention team at a Singapore telecom
    (think Singtel or StarHub). Each day you observe a customer's health
    metrics and decide whether/how to intervene.

    State (4,): [satisfaction_score, usage_frequency, months_active, support_tickets]
      All normalised [0, 1]. satisfaction decays naturally; tickets accumulate.

    Actions (4):
      0 = do nothing (free)
      1 = discount offer (costs $1, boosts satisfaction + usage)
      2 = proactive support call (costs $0.50, reduces tickets + boosts satisfaction)
      3 = feature upgrade (costs $1.50, boosts usage)

    Reward: +10 for retaining a customer through the month, -5 for churn,
            +1 per day customer stays, minus intervention cost.

    Episode: 30 steps (one month of daily decisions for one customer).


In [ ]:
class ChurnPreventionEnv(gym.Env):

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(4,), dtype=np.float32
        )
        self.action_space = spaces.Discrete(4)
        self.max_steps = 30
        self.state = None
        self.step_count = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = self.np_random.uniform(0.2, 0.8, size=(4,)).astype(np.float32)
        self.step_count = 0
        return self.state.copy(), {}

    def step(self, action):
        self.step_count += 1
        satisfaction, usage, tenure, tickets = self.state

        # TODO: Implement action effects on customer state
        # Action 0 = do nothing (cost 0)
        # Action 1 = discount: satisfaction +0.1, usage +0.05, cost 1.0
        # Action 2 = support call: tickets -0.15, satisfaction +0.05, cost 0.5
        # Action 3 = feature upgrade: usage +0.1, cost 1.5
        # Hint: clip satisfaction and usage to [0, 1] using min()
        intervention_cost = 0.0
        if action == 1:  # discount
            satisfaction = # TODO: min(1.0, satisfaction + 0.1)
            usage = # TODO: min(1.0, usage + 0.05)
            intervention_cost = 1.0
        elif action == 2:  # support call
            tickets = # TODO: max(0.0, tickets - 0.15)
            satisfaction = # TODO: min(1.0, satisfaction + 0.05)
            intervention_cost = 0.5
        elif action == 3:  # feature upgrade
            usage = # TODO: min(1.0, usage + 0.1)
            intervention_cost = 1.5

        # TODO: Natural drift — satisfaction decays, tickets accumulate
        # Hint: satisfaction -= 0.02 + noise, usage drifts slightly, tickets grow
        satisfaction = # TODO: max(0.0, satisfaction - 0.02 + np_random.normal(0, 0.02))
        usage = # TODO: max(0.0, min(1.0, usage - 0.01 + np_random.normal(0, 0.02)))
        tickets = # TODO: max(0.0, min(1.0, tickets + 0.02 + np_random.normal(0, 0.01)))
        tenure = min(1.0, tenure + 1.0 / self.max_steps)

        self.state = np.array([satisfaction, usage, tenure, tickets], dtype=np.float32)

        # TODO: Compute churn probability and determine if customer churns
        # Hint: churn_prob = max(0.0, 0.3 - satisfaction * 0.4 + tickets * 0.3)
        # Hint: churned = self.np_random.random() < churn_prob
        churn_prob = # TODO
        churned = # TODO

        if churned:
            reward = -5.0
            terminated = True
        else:
            reward = 1.0 - intervention_cost
            terminated = False

        truncated = self.step_count >= self.max_steps
        if truncated and not terminated:
            reward += 10.0  # bonus for retaining customer through the month

        return self.state.copy(), reward, terminated, truncated, {}



### Environment 2: Portfolio Rebalancing (hedge fund)


SCENARIO: You manage a Singapore-domiciled fund investing in three
    asset classes: SGX equities, Singapore government bonds, and cash (SGD).

    State (6,): [weight_stocks, weight_bonds, weight_cash,
                 market_volatility, interest_rate, momentum]
    Actions (27): 3^3 = all combinations of (decrease/hold/increase) for
                  each of the 3 assets. Weights are re-normalised after.
    Reward: portfolio_return - 0.5 * volatility_penalty - transaction_cost
    Episode: 24 steps (monthly rebalancing over 2 years).


Decode flat action to per-asset decisions: 0=decrease, 1=hold, 2=increase.


In [ ]:
class PortfolioRebalancingEnv(gym.Env):

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(6,), dtype=np.float32
        )
        self.action_space = spaces.Discrete(27)
        self.max_steps = 24
        self.state = None
        self.step_count = 0

    def _decode_action(self, action: int) -> list[int]:
        return [(action // (3**i)) % 3 for i in range(3)]

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        weights = np.array([0.4, 0.4, 0.2], dtype=np.float32)
        market = self.np_random.uniform(0.2, 0.5)
        rate = self.np_random.uniform(0.3, 0.6)
        momentum = 0.5
        self.state = np.concatenate([weights, [market, rate, momentum]]).astype(
            np.float32
        )
        self.step_count = 0
        return self.state.copy(), {}

    def step(self, action):
        self.step_count += 1
        weights = self.state[:3].copy()
        market_vol, interest, momentum = self.state[3], self.state[4], self.state[5]

        # TODO: Decode action and apply weight shifts
        # Hint: decisions = self._decode_action(action)
        # Hint: shifts = [[-0.05, 0.0, 0.05][d] for d in decisions]
        # Hint: transaction_cost = 0.005 * sum(abs(shifts))
        # Hint: weights = clip(weights + shifts, 0, 1) then normalise
        decisions = # TODO
        shifts = # TODO
        transaction_cost = # TODO
        weights = np.clip(weights + np.array(shifts, dtype=np.float32), 0.0, 1.0)
        weights = weights / (weights.sum() + 1e-8)

        # TODO: Simulate asset returns
        # Hint: stock_return = np_random.normal(0.01 + momentum * 0.02, market_vol * 0.1)
        # Hint: bond_return = np_random.normal(interest * 0.005, 0.02)
        # Hint: cash_return = 0.001
        # Hint: portfolio_return = dot(weights, [stock, bond, cash])
        stock_return = # TODO
        bond_return = # TODO
        cash_return = 0.001
        asset_returns = np.array([stock_return, bond_return, cash_return])
        portfolio_return = # TODO

        vol_penalty = market_vol * 0.05 * weights[0]
        reward = portfolio_return - vol_penalty - transaction_cost

        market_vol = np.clip(market_vol + self.np_random.normal(0, 0.03), 0.1, 0.9)
        interest = np.clip(interest + self.np_random.normal(0, 0.02), 0.1, 0.9)
        momentum = np.clip(momentum + self.np_random.normal(0, 0.1), 0.0, 1.0)

        self.state = np.concatenate([weights, [market_vol, interest, momentum]]).astype(
            np.float32
        )
        truncated = self.step_count >= self.max_steps
        return self.state.copy(), reward, False, truncated, {}



### Environment 3: Queue Management (Changi Airport)


SCENARIO: You manage immigration counter staffing at Changi Airport.
    Flights arrive in waves; you redistribute staff across three zones
    (T1, T2, T3) every 30 minutes.

    State (6,): [queue_t1, queue_t2, queue_t3, staff_t1, staff_t2, staff_t3]
      Queues normalised by capacity; staff normalised by total headcount.
    Actions (7): 6 shift pairs (move staff between zones) + do nothing
    Reward: -wait_time_penalty + service_bonus - reallocation_cost
    Episode: 48 steps (24-hour shift in 30-min intervals).


In [ ]:
class QueueManagementEnv(gym.Env):

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(6,), dtype=np.float32
        )
        self.action_space = spaces.Discrete(7)
        self.max_steps = 48
        self.state = None
        self.step_count = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        queues = self.np_random.uniform(0.1, 0.3, size=3).astype(np.float32)
        staff = np.array([0.33, 0.33, 0.34], dtype=np.float32)
        self.state = np.concatenate([queues, staff]).astype(np.float32)
        self.step_count = 0
        return self.state.copy(), {}

    def step(self, action):
        self.step_count += 1
        queues = self.state[:3].copy()
        staff = self.state[3:].copy()

        # TODO: Reallocate staff based on action (0-5 = shift pairs, 6 = do nothing)
        # Hint: shift_pairs = [(0,1),(0,2),(1,0),(1,2),(2,0),(2,1)]
        # Hint: if action < 6: move 0.08 staff from src to dst, cost = 0.15
        shift_pairs = [(0, 1), (0, 2), (1, 0), (1, 2), (2, 0), (2, 1)]
        realloc_cost = 0.0
        if action < 6:
            src, dst = shift_pairs[action]
            amount = min(0.08, staff[src])
            staff[src] -= amount
            staff[dst] += amount
            realloc_cost = 0.15

        # Flight arrival waves (Changi pattern: peaks at 6am, 12pm, 6pm, 11pm)
        half_hour = self.step_count % 48
        hour = half_hour / 2.0
        wave_t1 = 0.15 * np.exp(-0.5 * ((hour - 6) / 2) ** 2) + 0.1 * np.exp(
            -0.5 * ((hour - 18) / 2) ** 2
        )
        wave_t2 = 0.12 * np.exp(-0.5 * ((hour - 12) / 2) ** 2) + 0.08 * np.exp(
            -0.5 * ((hour - 23) / 2) ** 2
        )
        wave_t3 = 0.1 * np.exp(-0.5 * ((hour - 8) / 2) ** 2) + 0.12 * np.exp(
            -0.5 * ((hour - 20) / 2) ** 2
        )

        arrivals = np.array([wave_t1, wave_t2, wave_t3]) + self.np_random.normal(
            0, 0.02, size=3
        )
        arrivals = np.clip(arrivals, 0, 1)

        # TODO: Queue dynamics — arrivals add, staff serving removes
        # Hint: service_rate = staff * 0.3
        # Hint: queues = clip(queues + arrivals - service_rate, 0, 1)
        service_rate = # TODO
        queues = # TODO

        # TODO: Compute reward
        # Hint: avg_queue = mean(queues), max_queue = max(queues)
        # Hint: wait_penalty = 2.0 * avg_queue + 3.0 * max_queue
        # Hint: service_bonus = 1.0 if max_queue < 0.3 else 0.0
        avg_queue = # TODO
        max_queue = # TODO
        wait_penalty = # TODO
        service_bonus = # TODO
        reward = service_bonus - wait_penalty - realloc_cost

        self.state = np.concatenate([queues, staff]).astype(np.float32)
        truncated = self.step_count >= self.max_steps
        return self.state.copy(), reward, False, truncated, {}



### Environment 4: Energy Trading (SP Group)


SCENARIO: You manage the trading desk at SP Group. Every hour you
    decide whether to buy, sell, or hold electricity based on price
    forecasts, current reserves, and demand patterns.

    State (5,): [spot_price, reserve_level, demand_forecast, solar_output, time_of_day]
    Actions (5): 0=sell_large, 1=sell_small, 2=hold, 3=buy_small, 4=buy_large
    Reward: trading_profit - reserve_penalty
    Episode: 168 steps (hourly decisions for one week).


In [ ]:
class EnergyTradingEnv(gym.Env):

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(5,), dtype=np.float32
        )
        self.action_space = spaces.Discrete(5)
        self.max_steps = 168
        self.state = None
        self.step_count = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = np.array(
            [0.5, 0.5, 0.4, 0.0, 0.0], dtype=np.float32  # midnight start
        )
        self.step_count = 0
        return self.state.copy(), {}

    def step(self, action):
        self.step_count += 1
        price, reserve, demand, solar, time_of_day = self.state

        # TODO: Execute trading action
        # Hint: trade_amounts = [-0.15, -0.05, 0.0, 0.05, 0.15]
        # Hint: trade_cost = abs(trade) * 0.01
        # If buying (trade > 0): reserve += trade, pnl = -trade * price
        # If selling (trade < 0): actual_sell = min(-trade, reserve), pnl = actual_sell * price
        trade_amounts = [-0.15, -0.05, 0.0, 0.05, 0.15]
        trade = trade_amounts[action]
        trade_cost = abs(trade) * 0.01

        if trade > 0:
            reserve = # TODO
            trading_pnl = # TODO
        elif trade < 0:
            actual_sell = # TODO
            reserve = # TODO
            trading_pnl = # TODO
        else:
            trading_pnl = 0.0

        # Consume reserves to meet demand
        consumption = demand * 0.08
        unmet = max(0.0, consumption - reserve)
        reserve = max(0.0, reserve - consumption)

        # Penalties
        reserve_penalty = 3.0 * unmet
        low_reserve_penalty = 1.0 if reserve < 0.15 else 0.0

        reward = trading_pnl - trade_cost - reserve_penalty - low_reserve_penalty

        # Advance time
        time_of_day = (time_of_day + 1.0 / 24.0) % 1.0
        hour = time_of_day * 24

        # Price dynamics (Singapore: peaks at 12-3pm and 7-9pm)
        afternoon_peak = 0.2 * np.exp(-0.5 * ((hour - 14) / 2) ** 2)
        evening_peak = 0.15 * np.exp(-0.5 * ((hour - 20) / 2) ** 2)
        price = np.clip(
            0.3 + afternoon_peak + evening_peak + self.np_random.normal(0, 0.05),
            0.1,
            1.0,
        )

        # Solar output (tropical: peaks at noon, zero at night)
        if 7 <= hour <= 19:
            solar = np.clip(
                0.5 * np.sin(np.pi * (hour - 7) / 12) + self.np_random.normal(0, 0.05),
                0.0,
                1.0,
            )
        else:
            solar = 0.0
        reserve = min(1.0, reserve + solar * 0.03)

        # Demand forecast
        demand = np.clip(
            0.3
            + afternoon_peak * 0.8
            + evening_peak * 0.6
            + self.np_random.normal(0, 0.04),
            0.1,
            1.0,
        )

        self.state = np.array(
            [price, reserve, demand, solar, time_of_day], dtype=np.float32
        )
        truncated = self.step_count >= self.max_steps
        return self.state.copy(), reward, False, truncated, {}



### Environment 5: Traffic Signal Optimisation (LTA)


SCENARIO: You manage a 4-way intersection for the Land Transport
    Authority (LTA). Every cycle (90 seconds) you allocate green time
    between the north-south and east-west directions.

    State (4,): [queue_ns, queue_ew, flow_ns, flow_ew]
      Queues = vehicles waiting; flow = vehicles per minute arriving.
    Actions (5): green time allocation to NS direction
      0=20%, 1=35%, 2=50%, 3=65%, 4=80% (EW gets the remainder)
    Reward: -total_wait_time - queue_overflow_penalty
    Episode: 60 steps (90 minutes of signal cycles, ~1.5 hour rush).


In [ ]:
class TrafficSignalEnv(gym.Env):

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(4,), dtype=np.float32
        )
        self.action_space = spaces.Discrete(5)
        self.max_steps = 60
        self.state = None
        self.step_count = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = np.array([0.2, 0.2, 0.4, 0.4], dtype=np.float32)
        self.step_count = 0
        return self.state.copy(), {}

    def step(self, action):
        self.step_count += 1
        queue_ns, queue_ew, flow_ns, flow_ew = self.state

        # TODO: Allocate green time and compute service
        # Hint: ns_green_fraction = [0.20, 0.35, 0.50, 0.65, 0.80][action]
        # Hint: ew_green_fraction = 1.0 - ns_green_fraction
        # Hint: ns_served = min(queue_ns, ns_green_fraction * 0.5)
        # Hint: ew_served = min(queue_ew, ew_green_fraction * 0.5)
        ns_green_fraction = # TODO
        ew_green_fraction = # TODO

        ns_served = # TODO
        ew_served = # TODO
        queue_ns = max(0.0, queue_ns - ns_served)
        queue_ew = max(0.0, queue_ew - ew_served)

        # New arrivals (rush hour pattern)
        progress = self.step_count / self.max_steps
        rush_factor = 1.0 + 0.5 * np.sin(np.pi * progress)

        flow_ns = np.clip(0.3 * rush_factor + self.np_random.normal(0, 0.05), 0.1, 1.0)
        flow_ew = np.clip(0.35 * rush_factor + self.np_random.normal(0, 0.05), 0.1, 1.0)

        queue_ns = np.clip(queue_ns + flow_ns * 0.15, 0.0, 1.0)
        queue_ew = np.clip(queue_ew + flow_ew * 0.15, 0.0, 1.0)

        # Reward
        total_wait = queue_ns + queue_ew
        overflow_penalty = 2.0 * max(0.0, queue_ns - 0.8) + 2.0 * max(
            0.0, queue_ew - 0.8
        )
        reward = -total_wait - overflow_penalty

        self.state = np.array([queue_ns, queue_ew, flow_ns, flow_ew], dtype=np.float32)
        truncated = self.step_count >= self.max_steps
        return self.state.copy(), reward, False, truncated, {}



### Validate all environments against Gymnasium API


In [ ]:
env_classes = [
    ("ChurnPrevention", ChurnPreventionEnv),
    ("PortfolioRebalancing", PortfolioRebalancingEnv),
    ("QueueManagement", QueueManagementEnv),
    ("EnergyTrading", EnergyTradingEnv),
    ("TrafficSignal", TrafficSignalEnv),
]
for env_name, env_cls in env_classes:
    test_env = env_cls()
    obs, info = test_env.reset(seed=42)
    assert (
        obs.shape == test_env.observation_space.shape
    ), f"{env_name}: obs shape mismatch"
    obs2, reward, terminated, truncated, info = test_env.step(
        test_env.action_space.sample()
    )
    assert (
        obs2.shape == test_env.observation_space.shape
    ), f"{env_name}: step obs shape mismatch"
    assert isinstance(reward, (int, float)) or hasattr(reward, "__float__"), (
        f"{env_name}: reward should be numeric, got {type(reward).__name__}: {reward!r}"
    )
    print(
        f"  {env_name}: obs={obs.shape}, actions={test_env.action_space.n}, reward={reward:.3f}"
    )
    test_env.close()



### Checkpoint 1


In [ ]:
# INTERPRETATION: Each environment models a real business decision:
# - ChurnPrevention: when to intervene (cost of action vs cost of losing customer)
# - PortfolioRebalancing: asset allocation (risk-return tradeoff, transaction costs)
# - QueueManagement: staff allocation (service quality vs disruption cost)
# - EnergyTrading: buy/sell timing (price forecasting, reserve management)
# - TrafficSignal: green time allocation (competing demands, queue overflow)
# All follow the Gymnasium API: reset() -> (obs, info), step(a) -> (obs, r, term, trunc, info)
print("--- Checkpoint 1 passed --- all 5 custom environments validated\n")



## TASK 3 — Train DQN on ChurnPrevention, register in ModelRegistry


Train DQN on ChurnPrevention under a tracker.track(...) context.


In [ ]:
print("=" * 70)
print("  TASK 3: Train DQN on ChurnPrevention Environment")
print("=" * 70)

churn_env = ChurnPreventionEnv()
churn_obs_dim = churn_env.observation_space.shape[0]
churn_n_actions = churn_env.action_space.n

churn_dqn = DQN(churn_obs_dim, churn_n_actions).to(device)
churn_target = DQN(churn_obs_dim, churn_n_actions).to(device)
churn_target.load_state_dict(churn_dqn.state_dict())
churn_target.eval()

churn_opt = torch.optim.Adam(churn_dqn.parameters(), lr=1e-3)
churn_replay = ReplayBuffer(capacity=10_000)
churn_rewards_hist: list[float] = []
churn_actions_hist: list[list[int]] = []  # track action distribution per episode


async def _train_churn_dqn_async():
    churn_epsilon = 1.0

    async with tracker.track(
        experiment=exp_name, run_name="dqn_churn_prevention"
    ) as run:
        await run.log_params(
            {
                "algorithm": "DQN",
                "environment": "ChurnPrevention",
                "gamma": "0.99",
                "episodes": "150",
            }
        )

        print("== Training DQN on ChurnPrevention ==")
        for ep in range(150):
            state, _ = churn_env.reset(seed=ep)
            total_reward = 0.0
            done = False
            ep_actions: list[int] = []

            while not done:
                # TODO: Epsilon-greedy action selection (same pattern as 01_dqn.py)
                # Hint: random action with probability churn_epsilon, else argmax of churn_dqn
                if random.random() < churn_epsilon:
                    action = # TODO
                else:
                    with torch.no_grad():
                        s_t = torch.tensor(state, dtype=torch.float32, device=device)
                        action = # TODO

                ep_actions.append(action)
                next_state, reward, terminated, truncated, _ = churn_env.step(action)
                done = terminated or truncated
                churn_replay.push(state, action, reward, next_state, done)
                state = next_state
                total_reward += reward

                # TODO: Train on minibatch when replay has enough samples
                # Hint: same DQN training pattern — sample, compute Q-values,
                #   compute targets with churn_target, MSE loss, backprop
                if len(churn_replay) >= 300:
                    s_b, a_b, r_b, ns_b, d_b = churn_replay.sample(64)
                    q_vals = # TODO: churn_dqn(s_b).gather(1, a_b.unsqueeze(1)).squeeze(1)
                    with torch.no_grad():
                        next_q = # TODO: churn_target(ns_b).max(dim=1).values
                        targets = # TODO: r_b + 0.99 * next_q * (1.0 - d_b)
                    loss = # TODO: F.mse_loss(q_vals, targets)
                    churn_opt.zero_grad()
                    loss.backward()
                    churn_opt.step()

            churn_epsilon = max(0.01, churn_epsilon * 0.99)
            if (ep + 1) % 10 == 0:
                churn_target.load_state_dict(churn_dqn.state_dict())

            churn_rewards_hist.append(total_reward)
            churn_actions_hist.append(ep_actions)
            await run.log_metric("episode_reward", total_reward, step=ep)

            if (ep + 1) % 30 == 0:
                avg_30 = float(np.mean(churn_rewards_hist[-30:]))
                print(
                    f"  ep {ep+1:3d}  reward={total_reward:6.1f}  "
                    f"avg30={avg_30:6.1f}  eps={churn_epsilon:.3f}"
                )

        await run.log_metric(
            "final_avg_reward", float(np.mean(churn_rewards_hist[-30:]))
        )


await _train_churn_dqn_async()

# Register trained model
if has_registry:
    register_rl_model(
        registry,
        "m5_dqn_churn_prevention",
        churn_dqn,
        {
            "avg_reward_last30": float(np.mean(churn_rewards_hist[-30:])),
            "episodes_trained": 150.0,
        },
    )



### Checkpoint 2


In [ ]:
assert len(churn_rewards_hist) == 150, "Churn DQN should train for 150 episodes"
print("--- Checkpoint 2 passed --- DQN trained on ChurnPrevention\n")



## TASK 4 — Visualise: state trajectories, action distributions,
          learned policy vs baseline


In [ ]:
print("=" * 70)
print("  TASK 4: Visualise Custom Environment Behaviour")
print("=" * 70)

viz = ModelVisualizer()



### Plot 1: ChurnPrevention training reward curve


In [ ]:
# TODO: Plot training reward curve with moving average using viz.training_history()
fig1 = # TODO
fig1.write_html(str(OUTPUT_DIR / "03_churn_training_curve.html"))
print(f"  Saved: {OUTPUT_DIR / '03_churn_training_curve.html'}")



### Plot 2: Action distribution evolution


In [ ]:
# Compare action distributions: early training vs late training
action_names = ["Nothing", "Discount", "Support Call", "Feature Upgrade"]

early_actions = []
for ep_acts in churn_actions_hist[:20]:
    early_actions.extend(ep_acts)
late_actions = []
for ep_acts in churn_actions_hist[-20:]:
    late_actions.extend(ep_acts)

early_counts = [early_actions.count(i) / max(len(early_actions), 1) for i in range(4)]
late_counts = [late_actions.count(i) / max(len(late_actions), 1) for i in range(4)]

# TODO: Create grouped bar chart comparing early vs late action distributions
# Hint: go.Figure with two go.Bar traces — one for early, one for late
fig2 = # TODO
fig2.write_html(str(OUTPUT_DIR / "03_churn_action_distribution.html"))
print(f"  Saved: {OUTPUT_DIR / '03_churn_action_distribution.html'}")
# INTERPRETATION: Early training shows near-uniform action distribution
# (random exploration). Late training shows the agent has learned WHEN
# to intervene — it should use "nothing" when the customer is healthy
# and targeted interventions when churn risk is high.



### Plot 3: State trajectory of a single episode (trained agent)


In [ ]:
churn_env_viz = ChurnPreventionEnv()
state, _ = churn_env_viz.reset(seed=99)
trajectory = {"step": [], "satisfaction": [], "usage": [], "tickets": [], "action": []}

done = False
step = 0
while not done:
    with torch.no_grad():
        s_t = torch.tensor(state, dtype=torch.float32, device=device)
        action = int(churn_dqn(s_t).argmax().item())
    trajectory["step"].append(step)
    trajectory["satisfaction"].append(float(state[0]))
    trajectory["usage"].append(float(state[1]))
    trajectory["tickets"].append(float(state[3]))
    trajectory["action"].append(action_names[action])
    state, _, terminated, truncated, _ = churn_env_viz.step(action)
    done = terminated or truncated
    step += 1

traj_df = pl.DataFrame(trajectory)

# TODO: Create a 2-row subplot — row 1: state trajectories (satisfaction, usage, tickets)
#   row 2: action timeline as coloured bars
# Hint: make_subplots(rows=2, cols=1, shared_xaxes=True)
# Hint: add go.Scatter traces for satisfaction/usage/tickets in row 1
# Hint: add go.Bar traces for action colours in row 2
fig3 = # TODO
fig3.write_html(str(OUTPUT_DIR / "03_churn_episode_trajectory.html"))
print(f"  Saved: {OUTPUT_DIR / '03_churn_episode_trajectory.html'}")

churn_env_viz.close()



### Checkpoint 3


In [ ]:
assert Path(OUTPUT_DIR / "03_churn_training_curve.html").exists()
assert Path(OUTPUT_DIR / "03_churn_action_distribution.html").exists()
assert Path(OUTPUT_DIR / "03_churn_episode_trajectory.html").exists()
print("--- Checkpoint 3 passed --- environment visualisations generated\n")



## TASK 5 — Apply: Evaluate ChurnPrevention with business metrics


In [ ]:
print("=" * 70)
print("  TASK 5: Business Impact — ChurnPrevention Policy Evaluation")
print("=" * 70)



### Baseline: always do nothing


In [ ]:
def do_nothing_policy(state):
    return 0



### Baseline: always discount (aggressive retention)


In [ ]:
def always_discount_policy(state):
    return 1



### Learned DQN policy


Evaluate a churn policy and return detailed metrics.


In [ ]:
def dqn_churn_policy(state):
    with torch.no_grad():
        s = torch.tensor(state, dtype=torch.float32, device=device)
        return int(churn_dqn(s).argmax().item())


def evaluate_churn_policy(env_cls, policy_fn, n_episodes):
    results = {
        "rewards": [],
        "churned": [],
        "steps_survived": [],
        "intervention_count": [],
    }
    for i in range(n_episodes):
        env = env_cls()
        state, _ = env.reset(seed=2000 + i)
        total_reward = 0.0
        interventions = 0
        done = False
        steps = 0
        while not done:
            action = policy_fn(state)
            if action > 0:
                interventions += 1
            state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            done = terminated or truncated
            steps += 1
        results["rewards"].append(total_reward)
        results["churned"].append(terminated)
        results["steps_survived"].append(steps)
        results["intervention_count"].append(interventions)
        env.close()
    return results


# Run 100 episodes for statistical significance
n_eval = 100
nothing_eval = evaluate_churn_policy(ChurnPreventionEnv, do_nothing_policy, n_eval)
discount_eval = evaluate_churn_policy(
    ChurnPreventionEnv, always_discount_policy, n_eval
)
dqn_eval = evaluate_churn_policy(ChurnPreventionEnv, dqn_churn_policy, n_eval)

# Business metrics
print(
    f"\n  {'Policy':<25} {'Churn Rate':>12} {'Avg Reward':>12} {'Interventions':>14} {'Avg Survival':>14}"
)
print(f"  {'-'*77}")
for name, results in [
    ("Do Nothing", nothing_eval),
    ("Always Discount", discount_eval),
    ("DQN Learned", dqn_eval),
]:
    churn_rate = sum(results["churned"]) / len(results["churned"]) * 100
    avg_reward = np.mean(results["rewards"])
    avg_interventions = np.mean(results["intervention_count"])
    avg_survival = np.mean(results["steps_survived"])
    print(
        f"  {name:<25} {churn_rate:>10.1f}% {avg_reward:>12.1f} {avg_interventions:>14.1f} {avg_survival:>12.1f}d"
    )

# Revenue impact calculation
monthly_revenue_per_customer = 50.0  # SGD (typical telecom ARPU)
intervention_cost_per_action = 5.0  # SGD average

for name, results in [
    ("Do Nothing", nothing_eval),
    ("Always Discount", discount_eval),
    ("DQN Learned", dqn_eval),
]:
    retention_rate = 1.0 - sum(results["churned"]) / len(results["churned"])
    avg_interventions = np.mean(results["intervention_count"])
    retained_revenue = retention_rate * monthly_revenue_per_customer
    total_cost = avg_interventions * intervention_cost_per_action
    net_value = retained_revenue - total_cost
    print(f"\n  {name}:")
    print(f"    Retention rate: {retention_rate*100:.1f}%")
    print(f"    Revenue retained: SGD {retained_revenue:.2f}/customer/month")
    print(f"    Intervention cost: SGD {total_cost:.2f}/customer/month")
    print(f"    Net value: SGD {net_value:.2f}/customer/month")



### Comparison visualisation


In [ ]:
all_rewards = nothing_eval["rewards"] + discount_eval["rewards"] + dqn_eval["rewards"]
all_labels = (
    ["Do Nothing"] * n_eval + ["Always Discount"] * n_eval + ["DQN Learned"] * n_eval
)
eval_df = pl.DataFrame({"Policy": all_labels, "Monthly Reward": all_rewards})
# TODO: Create box plot comparing the three policies
# Hint: viz.box_plot(eval_df, "Monthly Reward", group_by="Policy")
fig_eval = # TODO
fig_eval.write_html(str(OUTPUT_DIR / "03_churn_business_impact.html"))
print(f"\n  Saved: {OUTPUT_DIR / '03_churn_business_impact.html'}")

# INTERPRETATION: The DQN learns to intervene SELECTIVELY — only when
# churn risk is high, and with the most cost-effective action. "Always
# Discount" has good retention but high cost. "Do Nothing" has low cost
# but high churn. The DQN finds the sweet spot: high retention, moderate
# cost, maximum net value per customer.

churn_env.close()



### Checkpoint 4


In [ ]:
assert len(nothing_eval["rewards"]) == n_eval
assert len(dqn_eval["rewards"]) == n_eval
assert Path(OUTPUT_DIR / "03_churn_business_impact.html").exists()
print("--- Checkpoint 4 passed --- business impact evaluation complete\n")


# Clean up
await conn.close()



## REFLECTION


[x] Built 5 Gymnasium-compliant environments for real business problems:
      1. ChurnPrevention (Singtel/StarHub) — customer retention interventions
      2. PortfolioRebalancing (hedge fund) — risk-adjusted asset allocation
      3. QueueManagement (Changi Airport) — staff allocation to counters
      4. EnergyTrading (SP Group) — electricity spot market trading
      5. TrafficSignal (LTA) — green light timing optimisation
  [x] Trained DQN on ChurnPrevention and registered in ModelRegistry
  [x] Visualised environment behaviour:
      - Training reward curves showing learning progress
      - Action distribution evolution (random -> strategic)
      - Single-episode trajectory with state + action timeline
  [x] Evaluated with business metrics:
      - Churn rate reduction vs baselines
      - Revenue retained per customer per month (SGD)
      - Net value: revenue minus intervention cost
      - DQN learns SELECTIVE intervention (best net value)

  KEY INSIGHT:
  The ENVIRONMENT is the hardest part of applied RL. Get the state,
  actions, and rewards right, and even simple algorithms (DQN) learn
  useful policies. Get them wrong, and even sophisticated algorithms
  learn the wrong thing.

  Next: In 04_algorithm_comparison.py, you'll compare Random vs DQN
  vs PPO side-by-side with a decision framework for which algorithm
  to use for which problem.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — Custom Environments")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — five instruments before Visualise


In [ ]:
# Reference: `kailash_ml.diagnostics` (via `kailash-ml`) — see gold standard
# `solutions/ex_1/01_standard_ae.py` for the full pattern.
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _diag_loss(m, batch):
    # Same as PPO/DQN — environment-specific reward
    # Customise per your exercise's loss shape.
    if isinstance(batch, (tuple, list)):
        x = batch[0]
        y = batch[1] if len(batch) > 1 else None
    else:
        x, y = batch, None
    out = m(x)
    import torch.nn.functional as F
    if y is None:
        return F.mse_loss(out, x)
    return F.cross_entropy(out, y)


print("\n── Diagnostic Report (Custom Gym Environment) ──")
try:
    diag, findings = run_diagnostic_checkpoint(
        agent,
        rollout_loader,
        _diag_loss,
        title="Custom Gym Environment",
        n_batches=8,
        show=False,
    )
except Exception as exc:
    # Diagnostic is pedagogical — never block the exercise on it.
    print(f"[diagnostic skipped: {exc}]")

# ══════ EXPECTED OUTPUT (synthesized reference — full run produces similar pattern) ══════



## DL Diagnostics Report — Prescription Pad


In [ ]:
# [✓] Gradient flow (HEALTHY): RMS in range, custom env reward well-scaled.
# [?] Warning: reward magnitude 1e+3 (high) — normalise for stable TD learning.



## STUDENT INTERPRETATION GUIDE — reading the Prescription Pad:


In [ ]:
#  [PRESCRIPTION] Custom environments often have poorly-scaled
#     rewards. If rewards are in the thousands, value estimates
#     explode and gradients blow up.
#     >> Prescription: normalise rewards to roughly [-1, 1] range
#        OR use reward clipping (env wrappers) OR adjust
#        discount factor gamma.
#
#  [STETHOSCOPE] Healthy gradient flow proves the environment is
#     learnable — the agent IS getting signal. Reward scaling is
#     an optimisation hygiene issue, not a design issue.

